# Drill: Web Scraping & Text bereinigen (LaptopPlatz)

Drei kurze Aufgaben auf Basis von `df_raw` der Übung «LaptopPlatz». Lege dieses Notebook **in denselben Ordner** wie `AP02_Uebung_LaptopPlatz.ipynb` (dort liegen `html/` und `laptops_clean.csv`).

**Vorgehen bei jeder Aufgabe (Teil-B-Checkliste):**
0. Ergebnis mit der Aufgabe vergleichen: Welche Zahl passt nicht?
1. Vermutung pro Auffälligkeit
2. Mit **einer** Zeile Code prüfen (Beispiele aus der Aufgabe einzeln testen)
3. Dort korrigieren, wo der Fehler **entsteht**
4. Mit den **Kontrollwerten** vergleichen

Die Hinweise sind eingeklappt. Öffne sie nur, wenn du nicht weiterkommst.

## Setup

In [1]:
import re
import pandas as pd
from bs4 import BeautifulSoup

# Scrape the three listing pages into df_raw (same as in the exercise)
rows = []
for page in [1, 2, 3]:
    with open(f'html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file.read(), 'html.parser')
    for card in soup.select('div.product-card'):
        rows.append([card['data-product-id'],
                     card.select('span.brand')[0].get_text().replace('\xa0', ' ').strip(),
                     card.select('span.model')[0].get_text().replace('\xa0', ' ').strip(),
                     card.select('li.ram')[0].get_text().replace('\xa0', ' ').strip(),
                     card.select('li.storage')[0].get_text().replace('\xa0', ' ').strip(),
                     card.select('li.color')[0].get_text().replace('\xa0', ' ').strip(),
                     card.select('span.price')[0].get_text().replace('\xa0', ' ').strip()])

df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'ram_raw',
                                     'storage_raw', 'color', 'price_raw'])
df_clean = pd.read_csv('laptops_clean.csv')

print('df_raw:', df_raw.shape, '| df_clean:', df_clean.shape)
df_raw.head()

df_raw: (130, 7) | df_clean: (120, 10)


,product_id,brand,model,ram_raw,storage_raw,color,price_raw
0,LP-20001,Dell,XPS 13,32 GB RAM,1 TB SSD,Graphit,CHF 2'149.–
1,LP-20002,ASUS,Zenbook 14 OLED,16 GB RAM,1 TB SSD,Blau,CHF 1'399.–
2,LP-20003,ASUS,ExpertBook B9,32 GB RAM,1 TB SSD,Schwarz,CHF 2'649.–
3,LP-20004,ASUS,ExpertBook B9,16 GB RAM,512 GB SSD,Schwarz,CHF 2'149.–
4,LP-20005,Microsoft,"Surface Laptop 7 13.8""",32 GB RAM,1 TB SSD,Saphir,CHF 1'949.–


---
## U1 – Fehleranalyse: Speicher in GB (8 Punkte)

**Ziel:** Speicherkapazität jedes Geräts in **GB** als Zahl (`storage_gb`). Es gilt **1 TB = 1024 GB** (wie in `laptops_clean.csv`). Gesucht: Anzahl Geräte, Median- und Durchschnittsspeicher sowie die Anzahl Geräte mit **mindestens 1000 GB**.

```python
df_U1 = df_raw[['product_id', 'storage_raw']]

storage = []
for s in df_U1['storage_raw']:
    storage.append(int(re.findall(r"\d+", s)[0]))

df_U1['storage_gb'] = storage

print('Anzahl Geräte:           ', df_U1.shape[0])
print('Median-Speicher (GB):    ', df_U1['storage_gb'].median())
print('Durchschnitt (GB):       ', round(df_U1['storage_gb'].mean(), 1))
print('Geräte mit >= 1000 GB:   ', (df_U1['storage_gb'] >= 1000).sum())
```

Tatsächliches Ergebnis:

```
Anzahl Geräte:            130
Median-Speicher (GB):     1.0
Durchschnitt (GB):        181.7
Geräte mit >= 1000 GB:    0
```

Der Shop führt **120 Artikel**. Beispiele für `storage_raw`: `256 GB SSD`, `512 GB SSD`, `1 TB SSD`.

**Aufgabe:** Korrigierter Code + Begründung jeder Änderung.

UNTERSCHEIDET NICHT ZWISCHEN TB UND GB

DUPLIKATE DRINN GEWESEN


In [37]:
df_U1 = df_raw[['product_id', 'storage_raw']]
df_U1 = df_U1.drop_duplicates().reset_index(drop=True)

storage = []
for s in df_U1['storage_raw']:
    d1 = re.findall(r'\d+',s)
    try:
        d2=int(d1[0])
        if 'TB' in s: 
            d2 = d2*1024
    except:
        d2=None

    storage.append(d2)
    
df_U1['storage_gb'] = pd.Series(storage, dtype='Int64')

print('Anzahl Geräte:           ', df_U1.shape[0])
print('Median-Speicher (GB):    ', df_U1['storage_gb'].median())
print('Durchschnitt (GB):       ', round(df_U1['storage_gb'].mean(), 2))
print('Geräte mit >= 1000 GB:   ', (df_U1['storage_gb'] >= 1000).sum())

Anzahl Geräte:            120
Median-Speicher (GB):     1024.0
Durchschnitt (GB):        706.13
Geräte mit >= 1000 GB:    62


<details><summary>Hinweise (nur bei Bedarf öffnen)</summary>

- Schritt 0: Ein Median von **1 GB** und **0** Geräte mit ≥ 1000 GB, obwohl es Geräte mit «1 TB» gibt. Was liest der Code bei `1 TB SSD`?
- Teste die drei Beispiele einzeln mit `re.findall(r"\d+", "1 TB SSD")`.
- Die Zahl allein reicht nicht: Du brauchst auch die **Einheit**. Wie prüfst du, ob ein Text `TB` enthält? (Tipp: `if 'TB' in s:`)
- Und die 130?
</details>

**Kontrollwerte:** 120 · 1024.0 · 706.1 · 62

---
## U2 – Fehleranalyse: Arbeitsspeicher (8 Punkte)

**Ziel:** Arbeitsspeicher als Zahl (`ram_gb`) und die Anzahl Geräte mit **mindestens 16 GB RAM**.

```python
df_U2 = df_raw[['product_id', 'ram_raw']]
df_U2['ram_gb'] = df_U2['ram_raw'].str.extract(r'(\d)').astype(int)

print('Anzahl Geräte:          ', df_U2.shape[0])
print('Geräte mit >= 16 GB:    ', (df_U2['ram_gb'] >= 16).sum())
print(df_U2['ram_gb'].value_counts())
```

Tatsächliches Ergebnis:

```
Anzahl Geräte:           130
Geräte mit >= 16 GB:     0
ram_gb
1    63
8    36
3    31
```

Beispiele für `ram_raw`: `8 GB RAM`, `16 GB RAM`, `32 GB RAM`.

**Aufgabe:** Korrigierter Code + Begründung jeder Änderung.

**Begründung der Änderungen:**

[Hier eintragen]

In [57]:
df_U2 = df_raw[['product_id', 'ram_raw']]
df_U2 = df_U2.drop_duplicates().reset_index(drop=True)

ram=[]
for r in df_U2['ram_raw']:
    d1 = re.findall(r'\d+',r)		#normal Ziffern suchen
    try:
        d2=int(d1[0])			#d in zahl ändern
    except:
        d2=None

    ram.append(d2)

df_U2['ram_gb'] = pd.Series(ram, dtype='Int64') #I gross von Int64 damit auch None angezeigt wird


print('Anzahl Geräte:          ', df_U2.shape[0])
print('Geräte mit >= 16 GB:    ', (df_U2['ram_gb'] >= 16).sum())
print(df_U2['ram_gb'].value_counts())

Anzahl Geräte:           120
Geräte mit >= 16 GB:     87
ram_gb
16    56
8     33
32    31
Name: count, dtype: Int64


<details><summary>Hinweise (nur bei Bedarf öffnen)</summary>

- Schritt 0: Es gibt RAM-Grössen **1** und **3** GB? Woher kommen die?
- Teste: `re.findall(r"\d", "16 GB RAM")` und `re.findall(r"\d+", "16 GB RAM")`. Was ist der Unterschied?
- `.str.extract(...)` ist die pandas-Variante von `re.findall` für eine ganze Spalte. Es nimmt den **ersten** Treffer der Klammer `( )`.
- Und die 130?
</details>

**Kontrollwerte:** 120 · 87 · value_counts: 16 → 56, 8 → 33, 32 → 31